# SVM — Fixed Percentile Subsampling
## Training and Evaluation (No Trends vs Google Trends)

## Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
import time
import warnings
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, PredefinedSplit
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

SCRIPT_DIR    = Path().resolve()
ARTIFACTS_DIR = SCRIPT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
PLOT_DIR      = ARTIFACTS_DIR / "plots"
PLOT_DIR.mkdir(exist_ok=True)
RANDOM_STATE  = 42

DATA_PATH = SCRIPT_DIR / "../../data/polymarket_ml_dataset_with_trends_clean.parquet"

PERCENTILE_POINTS = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
BEST_C            = 10.0
BEST_GAMMA        = 0.001

COLORS = {
    'no_trends':   'steelblue',
    'with_trends': 'seagreen',
    'baseline':    'coral',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

print(f"Data path:   {DATA_PATH.resolve()}")
print(f"File exists: {DATA_PATH.exists()}")

Data path:   /Users/ssastri1/Desktop/DS-GA-1003/Project/predicting-the-future/data/polymarket_ml_dataset_with_trends_clean.parquet
File exists: True


## Section 1: Load Data and Fixed Percentile Subsampling

In [ ]:
print("Loading dataset...")
df = pd.read_parquet(DATA_PATH, engine="fastparquet")
df['category'] = df['category'].fillna('unknown')
df['outcome']  = df['outcome'].astype(int)
df['snapshot_timestamp'] = pd.to_datetime(df['snapshot_timestamp'], format='ISO8601', utc=True)

print(f"Full dataset: {df.shape[0]:,} rows, {df['market_id'].nunique():,} markets")
print(f"Split balance: {df.groupby('split')['market_id'].nunique().to_dict()}")

def get_percentile_snapshots(group):
    group = group.sort_values('pct_lifetime_elapsed')
    selected = []
    for pct in PERCENTILE_POINTS:
        idx = (group['pct_lifetime_elapsed'] - pct).abs().idxmin()
        row = group.loc[idx].copy()
        row['target_percentile'] = pct
        selected.append(row)
    return pd.DataFrame(selected)

print(f"\nSubsampling to {len(PERCENTILE_POINTS)} fixed percentile snapshots per market...")

df_sampled = (
    df.groupby('market_id', group_keys=False)
    .apply(get_percentile_snapshots)
    .reset_index(drop=True)
)

rows_per_market = df_sampled.groupby('market_id').size()
print(f"\nDataset after subsampling:")
print(f"  Rows:    {df_sampled.shape[0]:,}  (was {df.shape[0]:,})")
print(f"  Markets: {df_sampled['market_id'].nunique():,}")
print(f"  Rows per market — min: {rows_per_market.min()}, max: {rows_per_market.max()}, mean: {rows_per_market.mean():.1f}")
print(f"  All markets have exactly {len(PERCENTILE_POINTS)} rows: {(rows_per_market == len(PERCENTILE_POINTS)).all()}")

print(f"\nActual vs target pct_lifetime_elapsed (mean across markets):")
actual_pcts = df_sampled.groupby('target_percentile')['pct_lifetime_elapsed'].mean()
for target, actual in actual_pcts.items():
    print(f"  target={target:.2f}  actual={actual:.3f}  diff={abs(actual-target):.3f}")

print(f"\nOutcome balance: {df_sampled['outcome'].value_counts(normalize=True).round(3).to_dict()}")
print(f"Split balance:   {df_sampled['split'].value_counts().to_dict()}")

sampled_path = ARTIFACTS_DIR / "df_sampled_clean.parquet"
df_sampled.to_parquet(sampled_path, index=False)
print(f"\nSaved to: {sampled_path} ({sampled_path.stat().st_size / (1024*1024):.1f} MB)")

Loading dataset...


OSError: Repetition level histogram size mismatch

## Section 2: Prepare Train/Test Splits

In [ ]:
# Load from disk so section 1 can be commented out after first run
df_sampled = pd.read_parquet(ARTIFACTS_DIR / "df_sampled_clean.parquet")
df_sampled['outcome']  = df_sampled['outcome'].astype(int)
df_sampled['category'] = df_sampled['category'].fillna('unknown')

assert df_sampled['outcome'].isna().sum() == 0, "NAs in outcome"
assert (df_sampled.groupby('market_id').size() == 7).all(), "Not all markets have 7 rows"

train_all = df_sampled[df_sampled['split'] == 'train'].copy().reset_index(drop=True)
test_all  = df_sampled[df_sampled['split'] == 'test'].copy().reset_index(drop=True)

print(f"Full split (all markets):")
print(f"  Train: {train_all.shape[0]:,} rows, {train_all['market_id'].nunique():,} markets")
print(f"  Test:  {test_all.shape[0]:,} rows,  {test_all['market_id'].nunique():,} markets")

# Trends subset — exclude markets with no trend data (other category + ~133 markets)
markets_with_trends = df_sampled.groupby('market_id')['has_trend_data'].max()
trend_market_ids    = markets_with_trends[markets_with_trends == 1].index

train_trends = train_all[train_all['market_id'].isin(trend_market_ids)].copy().reset_index(drop=True)
test_trends  = test_all[test_all['market_id'].isin(trend_market_ids)].copy().reset_index(drop=True)
test_shared  = test_all[test_all['market_id'].isin(trend_market_ids)].copy().reset_index(drop=True)

print(f"\nTrends subset (markets with trend data):")
print(f"  Train: {train_trends.shape[0]:,} rows, {train_trends['market_id'].nunique():,} markets")
print(f"  Test:  {test_trends.shape[0]:,} rows,  {test_trends['market_id'].nunique():,} markets")
print(f"  Markets dropped (no trend data): {train_all['market_id'].nunique() - train_trends['market_id'].nunique():,}")
print(f"  Categories remaining: {sorted(train_trends['category'].unique())}")

## Section 3: Define Feature Sets

In [ ]:
# Base features — shared by both models
# Excluded: price_mean_7d/14d (r>0.96 with price_at_snapshot),
#           pct_lifetime_elapsed (r=0.93 with target_percentile),
#           price_deviation_from_half (redundant), total_volume (redundant),
#           days_before_close (redundant), price_min/max/range (redundant)
base_numeric = [
    'price_at_snapshot',
    'target_percentile',
    'duration_days',
    'log_volume',
    'price_volatility_7d',
    'price_change_7d',
    'price_trend_7d',
    'price_volatility_14d',
    'price_change_14d',
    'price_trend_14d',
]

# Trends features — additional for trends model
# has_trend_data excluded — redundant with category (other=0, everything else=1)
trends_numeric = [
    'trend_value',       # google trends search interest (0-100 scale)
    'trend_ma4',         # 4-week moving average of trend_value
    'trend_change_4w',   # change in trend_value over past 4 weeks
    'trend_spike',       # binary: did search interest spike recently
]

categorical_features = ['category']
features_no_trends   = base_numeric + categorical_features
features_with_trends = base_numeric + trends_numeric + categorical_features
target               = 'outcome'

print(f"Non-trends model: {len(features_no_trends)} features ({len(base_numeric)} numeric + 1 categorical)")
print(f"Trends model:     {len(features_with_trends)} features ({len(base_numeric) + len(trends_numeric)} numeric + 1 categorical)")
print(f"Trends features added: {trends_numeric}")

# Correlation check — confirm no highly correlated pairs
corr_matrix = train_all[base_numeric].corr().round(2)
high_corr   = [(base_numeric[i], base_numeric[j], abs(corr_matrix.iloc[i, j]))
               for i in range(len(base_numeric))
               for j in range(i+1, len(base_numeric))
               if abs(corr_matrix.iloc[i, j]) > 0.85]

print(f"\nHighly correlated pairs (|r| > 0.85): {len(high_corr)}")
if high_corr:
    for f1, f2, c in sorted(high_corr, key=lambda x: -x[2]):
        print(f"  {f1} vs {f2}  r={c:.2f}")
else:
    print("  None — feature set is clean")

print(f"\nCorrelations of trend features with price_at_snapshot:")
for col in trends_numeric:
    corr = train_trends['price_at_snapshot'].corr(train_trends[col])
    print(f"  {col:25s}  r={corr:+.3f}")

print(f"\nMissing values in trend features (train_trends):")
print(train_trends[trends_numeric].isna().sum().to_string())

## Section 4: Handle Rare Categories

In [ ]:
RARE_THRESHOLD = 0.02

def collapse_rare_categories(train_df, test_df, threshold=RARE_THRESHOLD):
    """Collapse rare category levels into 'other' in both train and test."""
    freq = (
        train_df.drop_duplicates('market_id')[['market_id', 'category']]
        .groupby('category')['market_id'].count()
        .div(train_df['market_id'].nunique())
    )
    rare = freq[freq < threshold].index.tolist()
    train_df = train_df.copy()
    test_df  = test_df.copy()
    train_df['category'] = train_df['category'].replace(rare, 'other')
    test_df['category']  = test_df['category'].replace(rare, 'other')
    known = set(train_df['category'].unique())
    test_df['category'] = test_df['category'].apply(lambda x: x if x in known else 'other')
    return train_df, test_df, rare

train_all,    test_all,    rare_all    = collapse_rare_categories(train_all,    test_all)
train_trends, test_trends, rare_trends = collapse_rare_categories(train_trends, test_trends)
_,            test_shared, _           = collapse_rare_categories(train_all,    test_all[test_all['market_id'].isin(trend_market_ids)].copy())

print(f"Non-trends model:")
print(f"  Rare categories collapsed: {rare_all}")
print(f"  Final levels: {sorted(train_all['category'].unique())}")
print(f"  Outcome rate by category:")
print(train_all.drop_duplicates('market_id').groupby('category')['outcome'].mean().sort_values(ascending=False).round(3).to_string())

print(f"\nTrends model:")
print(f"  Rare categories collapsed: {rare_trends}")
print(f"  Final levels: {sorted(train_trends['category'].unique())}")

## Section 5: Build Preprocessing Pipelines

In [ ]:
def build_preprocessor(num_features, cat_features):
    """Build ColumnTransformer: median imputation + scaling for numeric,
    constant imputation + OHE for categorical."""
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer(transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ])

# Sanity check both preprocessors
for name, num_feats, train_df in [
    ('No-trends',   base_numeric,                  train_all),
    ('With-trends', base_numeric + trends_numeric, train_trends),
]:
    prep = build_preprocessor(num_feats, categorical_features)
    X    = train_df[num_feats + categorical_features]
    Xt   = prep.fit_transform(X)
    means = Xt[:, :len(num_feats)].mean(axis=0).round(3)
    stds  = Xt[:, :len(num_feats)].std(axis=0).round(3)
    print(f"{name}: input {X.shape} → transformed {Xt.shape}")
    print(f"  Post-scaling means (should be ~0): {means}")
    print(f"  Post-scaling stds  (should be ~1): {stds}")
    print()

## Section 6: Train Non-Trends SVM

In [ ]:
X_train_all = train_all[features_no_trends]
y_train_all = train_all[target]

svm_no_trends = Pipeline(steps=[
    ('preprocessor', build_preprocessor(base_numeric, categorical_features)),
    ('svm', SVC(
        kernel='rbf',
        C=BEST_C,
        gamma=BEST_GAMMA,
        probability=True,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

print(f"Training non-trends SVM on {train_all['market_id'].nunique():,} markets ({len(X_train_all):,} rows)...")
print(f"Params: C={BEST_C}, gamma={BEST_GAMMA}")

start = time.time()
svm_no_trends.fit(X_train_all, y_train_all)
elapsed = time.time() - start

n_support = svm_no_trends['svm'].n_support_
print(f"\nTraining completed in {elapsed/60:.1f} minutes")
print(f"Support vectors: {sum(n_support):,} ({sum(n_support)/len(X_train_all)*100:.1f}% of training rows)")
print(f"  Class 0: {n_support[0]:,}  Class 1: {n_support[1]:,}")

joblib.dump(svm_no_trends, ARTIFACTS_DIR / "svm_no_trends.pkl")
print(f"\nSaved: svm_no_trends.pkl ({(ARTIFACTS_DIR / 'svm_no_trends.pkl').stat().st_size / (1024*1024):.1f} MB)")

## Section 7: Train Trends SVM

In [ ]:
X_train_trends = train_trends[features_with_trends]
y_train_trends = train_trends[target]

svm_with_trends = Pipeline(steps=[
    ('preprocessor', build_preprocessor(base_numeric + trends_numeric, categorical_features)),
    ('svm', SVC(
        kernel='rbf',
        C=BEST_C,
        gamma=BEST_GAMMA,
        probability=True,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

print(f"Training trends SVM on {train_trends['market_id'].nunique():,} markets ({len(X_train_trends):,} rows)...")
print(f"Params: C={BEST_C}, gamma={BEST_GAMMA}")

start = time.time()
svm_with_trends.fit(X_train_trends, y_train_trends)
elapsed = time.time() - start

n_support = svm_with_trends['svm'].n_support_
print(f"\nTraining completed in {elapsed/60:.1f} minutes")
print(f"Support vectors: {sum(n_support):,} ({sum(n_support)/len(X_train_trends)*100:.1f}% of training rows)")
print(f"  Class 0: {n_support[0]:,}  Class 1: {n_support[1]:,}")

joblib.dump(svm_with_trends, ARTIFACTS_DIR / "svm_with_trends.pkl")
print(f"\nSaved: svm_with_trends.pkl ({(ARTIFACTS_DIR / 'svm_with_trends.pkl').stat().st_size / (1024*1024):.1f} MB)")

## Section 8: Load Models and Generate Predictions
*(Start here after training is complete)*

In [ ]:
# Load subsampled dataset
df_sampled = pd.read_parquet(ARTIFACTS_DIR / "df_sampled_clean.parquet")
df_sampled['outcome']  = df_sampled['outcome'].astype(int)
df_sampled['category'] = df_sampled['category'].fillna('unknown')

markets_with_trends = df_sampled.groupby('market_id')['has_trend_data'].max()
trend_market_ids    = markets_with_trends[markets_with_trends == 1].index

train_all_ref    = df_sampled[df_sampled['split'] == 'train'].copy()
train_trends_ref = train_all_ref[train_all_ref['market_id'].isin(trend_market_ids)].copy()

test_all    = df_sampled[df_sampled['split'] == 'test'].copy().reset_index(drop=True)
test_trends = test_all[test_all['market_id'].isin(trend_market_ids)].copy().reset_index(drop=True)
test_shared = test_all[test_all['market_id'].isin(trend_market_ids)].copy().reset_index(drop=True)

# Recreate feature lists
base_numeric = [
    'price_at_snapshot', 'target_percentile', 'duration_days', 'log_volume',
    'price_volatility_7d', 'price_change_7d', 'price_trend_7d',
    'price_volatility_14d', 'price_change_14d', 'price_trend_14d',
]
trends_numeric       = ['trend_value', 'trend_ma4', 'trend_change_4w', 'trend_spike']
categorical_features = ['category']
features_no_trends   = base_numeric + categorical_features
features_with_trends = base_numeric + trends_numeric + categorical_features
target               = 'outcome'
RARE_THRESHOLD       = 0.02

def collapse_rare_categories(train_df, test_df, threshold=RARE_THRESHOLD):
    freq = (
        train_df.drop_duplicates('market_id')[['market_id', 'category']]
        .groupby('category')['market_id'].count()
        .div(train_df['market_id'].nunique())
    )
    rare = freq[freq < threshold].index.tolist()
    df   = test_df.copy()
    df['category'] = df['category'].replace(rare, 'other')
    known = set(train_df['category'].unique())
    df['category'] = df['category'].apply(lambda x: x if x in known else 'other')
    return df

test_all    = collapse_rare_categories(train_all_ref,    test_all)
test_shared = collapse_rare_categories(train_all_ref,    test_shared)
test_trends = collapse_rare_categories(train_trends_ref, test_trends)

# Load models
svm_no_trends   = joblib.load(ARTIFACTS_DIR / "svm_no_trends.pkl")
svm_with_trends = joblib.load(ARTIFACTS_DIR / "svm_with_trends.pkl")
print("Models loaded: svm_no_trends.pkl ✓  svm_with_trends.pkl ✓")
print(f"\nTest sets:")
print(f"  Full:    {test_all.shape[0]:,} rows, {test_all['market_id'].nunique():,} markets")
print(f"  Shared:  {test_shared.shape[0]:,} rows, {test_shared['market_id'].nunique():,} markets")
print(f"  Trends:  {test_trends.shape[0]:,} rows, {test_trends['market_id'].nunique():,} markets")

# Generate predictions
y_pred_proba_nt_full   = svm_no_trends.predict_proba(test_all[features_no_trends])[:, 1]
y_pred_class_nt_full   = svm_no_trends.predict(test_all[features_no_trends])
y_pred_proba_nt_shared = svm_no_trends.predict_proba(test_shared[features_no_trends])[:, 1]
y_pred_proba_wt        = svm_with_trends.predict_proba(test_trends[features_with_trends])[:, 1]
y_pred_class_wt        = svm_with_trends.predict(test_trends[features_with_trends])

baseline_full   = test_all['price_at_snapshot'].clip(1e-6, 1 - 1e-6)
baseline_shared = test_shared['price_at_snapshot'].clip(1e-6, 1 - 1e-6)
baseline_trends = test_trends['price_at_snapshot'].clip(1e-6, 1 - 1e-6)

# Store predictions in dataframes for downstream use
test_all['pred_proba_nt']    = y_pred_proba_nt_full
test_all['baseline']         = baseline_full.values
test_shared['pred_proba_nt'] = y_pred_proba_nt_shared
test_shared['baseline']      = baseline_shared.values
test_trends['pred_proba_wt'] = y_pred_proba_wt
test_trends['baseline']      = baseline_trends.values

print(f"\nPredictions generated ✓")

## Section 9: Overall Metrics

In [ ]:
y_full   = test_all['outcome']
y_shared = test_shared['outcome']
y_trends = test_trends['outcome']

# Full test set — non-trends model
auc_nt_f = roc_auc_score(y_full,   y_pred_proba_nt_full)
bri_nt_f = brier_score_loss(y_full, y_pred_proba_nt_full)
auc_bl_f = roc_auc_score(y_full,   baseline_full)
bri_bl_f = brier_score_loss(y_full, baseline_full)

# Shared test set — apples-to-apples comparison
auc_nt_s = roc_auc_score(y_shared,  y_pred_proba_nt_shared)
bri_nt_s = brier_score_loss(y_shared, y_pred_proba_nt_shared)
auc_wt   = roc_auc_score(y_trends,  y_pred_proba_wt)
bri_wt   = brier_score_loss(y_trends, y_pred_proba_wt)
auc_bl_s = roc_auc_score(y_shared,  baseline_shared)
bri_bl_s = brier_score_loss(y_shared, baseline_shared)

print(f"── Full Test Set (all markets) ─────────────────────────────")
print(f"{'Metric':<20} {'SVM (no trends)':>18} {'Baseline':>12}")
print(f"{'-'*52}")
print(f"{'ROC-AUC':<20} {auc_nt_f:>18.4f} {auc_bl_f:>12.4f}")
print(f"{'Brier Score':<20} {bri_nt_f:>18.4f} {bri_bl_f:>12.4f}")
print(f"{'N markets':<20} {test_all['market_id'].nunique():>18,}")

print(f"\n── Shared Test Set (markets with trend data) ───────────────")
print(f"{'Metric':<20} {'SVM (no trends)':>18} {'SVM+Trends':>12} {'Baseline':>12}")
print(f"{'-'*64}")
print(f"{'ROC-AUC':<20} {auc_nt_s:>18.4f} {auc_wt:>12.4f} {auc_bl_s:>12.4f}")
print(f"{'Brier Score':<20} {bri_nt_s:>18.4f} {bri_wt:>12.4f} {bri_bl_s:>12.4f}")
print(f"{'N markets':<20} {test_shared['market_id'].nunique():>18,}")

print(f"\nTrends vs no-trends — ROC-AUC: {auc_wt - auc_nt_s:+.4f}, Brier: {bri_nt_s - bri_wt:+.4f}")
print(f"Trends vs baseline  — ROC-AUC: {auc_wt - auc_bl_s:+.4f}, Brier: {bri_bl_s - bri_wt:+.4f}")

print(f"\n── Classification Report: SVM (no trends, full test) ───────")
print(classification_report(y_full, y_pred_class_nt_full, target_names=['outcome=0', 'outcome=1']))

print(f"── Classification Report: SVM+Trends ───────────────────────")
print(classification_report(y_trends, y_pred_class_wt, target_names=['outcome=0', 'outcome=1']))

## Section 10: Stratified Evaluation

In [ ]:
percentiles = sorted(test_shared['target_percentile'].unique())

# By lifetime stage
print(f"── ROC-AUC by Lifetime Stage (shared test set) ─────────────────────────")
print(f"{'Percentile':<12} {'SVM (no trends)':>18} {'SVM+Trends':>12} {'Baseline':>12}")
print(f"{'-'*56}")

for pct in percentiles:
    sub_s = test_shared[test_shared['target_percentile'] == pct]
    sub_t = test_trends[test_trends['target_percentile'] == pct]
    auc_nt = roc_auc_score(sub_s['outcome'], sub_s['pred_proba_nt'])
    auc_wt = roc_auc_score(sub_t['outcome'], sub_t['pred_proba_wt'])
    auc_bl = roc_auc_score(sub_s['outcome'], sub_s['baseline'])
    print(f"{pct:<12.2f} {auc_nt:>18.4f} {auc_wt:>12.4f} {auc_bl:>12.4f}")

# By category
print(f"\n── ROC-AUC by Category (shared test set) ───────────────────────────────")
print(f"{'Category':<20} {'SVM (no trends)':>18} {'SVM+Trends':>12} {'Baseline':>12} {'Markets':>10}")
print(f"{'-'*62}")

categories = sorted(test_shared['category'].unique())
cat_aucs_nt, cat_aucs_wt, cat_aucs_bl = [], [], []

for cat in categories:
    sub_s = test_shared[test_shared['category'] == cat]
    sub_t = test_trends[test_trends['category'] == cat]
    try:
        a_nt = roc_auc_score(sub_s['outcome'], sub_s['pred_proba_nt'])
        a_wt = roc_auc_score(sub_t['outcome'], sub_t['pred_proba_wt'])
        a_bl = roc_auc_score(sub_s['outcome'], sub_s['baseline'])
    except ValueError:
        a_nt = a_wt = a_bl = float('nan')
    cat_aucs_nt.append(a_nt)
    cat_aucs_wt.append(a_wt)
    cat_aucs_bl.append(a_bl)
    n_mkts = sub_s['market_id'].nunique()
    print(f"{cat:<20} {a_nt:>18.4f} {a_wt:>12.4f} {a_bl:>12.4f} {n_mkts:>10,}")

# Market-weighted accuracy
test_all['correct_nt']    = (test_all['pred_proba_nt'].round() == test_all['outcome']).astype(int)
test_trends['correct_wt'] = (test_trends['pred_proba_wt'].round() == test_trends['outcome']).astype(int)

print(f"\n── Market-Weighted Accuracy ────────────────────────────────────────────")
print(f"  SVM (no trends): {test_all.groupby('market_id')['correct_nt'].mean().mean():.4f}")
print(f"  SVM+Trends:      {test_trends.groupby('market_id')['correct_wt'].mean().mean():.4f}")

## Section 11: Plots

In [ ]:
# ── Plot 1: ROC Curves ──────────────────────────────────────────────────────────
print("Generating Plot 1: ROC Curves...")
fpr_nt, tpr_nt, _ = roc_curve(y_shared, y_pred_proba_nt_shared)
fpr_wt, tpr_wt, _ = roc_curve(y_trends, y_pred_proba_wt)
fpr_bl, tpr_bl, _ = roc_curve(y_shared, baseline_shared)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_nt, tpr_nt, label=f'SVM no trends (AUC={auc_nt_s:.4f})', color=COLORS['no_trends'],   lw=2)
ax.plot(fpr_wt, tpr_wt, label=f'SVM+Trends (AUC={auc_wt:.4f})',      color=COLORS['with_trends'], lw=2)
ax.plot(fpr_bl, tpr_bl, label=f'Baseline (AUC={auc_bl_s:.4f})',       color=COLORS['baseline'],    lw=2, linestyle='--')
ax.plot([0, 1], [0, 1],  label='Random (AUC=0.50)',                    color='gray',                lw=1, linestyle=':')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Shared Test Set')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'roc_curve.png', dpi=150)
plt.show()
print("  Saved: roc_curve.png")

In [ ]:
# ── Plot 2: Precision-Recall Curves ─────────────────────────────────────────────
print("Generating Plot 2: Precision-Recall Curves...")
ap_nt = average_precision_score(y_shared, y_pred_proba_nt_shared)
ap_wt = average_precision_score(y_trends, y_pred_proba_wt)
ap_bl = average_precision_score(y_shared, baseline_shared)

prec_nt, rec_nt, _ = precision_recall_curve(y_shared, y_pred_proba_nt_shared)
prec_wt, rec_wt, _ = precision_recall_curve(y_trends, y_pred_proba_wt)
prec_bl, rec_bl, _ = precision_recall_curve(y_shared, baseline_shared)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(rec_nt, prec_nt, label=f'SVM no trends (AP={ap_nt:.4f})', color=COLORS['no_trends'],   lw=2)
ax.plot(rec_wt, prec_wt, label=f'SVM+Trends (AP={ap_wt:.4f})',    color=COLORS['with_trends'], lw=2)
ax.plot(rec_bl, prec_bl, label=f'Baseline (AP={ap_bl:.4f})',       color=COLORS['baseline'],    lw=2, linestyle='--')
ax.axhline(y_shared.mean(), color='gray', lw=1, linestyle=':', label=f'No-skill ({y_shared.mean():.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Shared Test Set')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'precision_recall_curve.png', dpi=150)
plt.show()
print("  Saved: precision_recall_curve.png")

In [ ]:
# ── Plot 3: Calibration ──────────────────────────────────────────────────────────
print("Generating Plot 3: Calibration Plot...")
pt_nt, pp_nt = calibration_curve(y_shared, y_pred_proba_nt_shared, n_bins=20)
pt_wt, pp_wt = calibration_curve(y_trends, y_pred_proba_wt,        n_bins=20)
pt_bl, pp_bl = calibration_curve(y_shared, baseline_shared,         n_bins=20)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(pp_nt, pt_nt, label='SVM no trends', color=COLORS['no_trends'],   lw=2, marker='o', markersize=4)
ax.plot(pp_wt, pt_wt, label='SVM+Trends',    color=COLORS['with_trends'], lw=2, marker='s', markersize=4)
ax.plot(pp_bl, pt_bl, label='Baseline',       color=COLORS['baseline'],    lw=2, marker='^', markersize=4, linestyle='--')
ax.plot([0, 1], [0, 1], label='Perfect',      color='gray',                lw=1, linestyle=':')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Plot — Shared Test Set')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'calibration.png', dpi=150)
plt.show()
print("  Saved: calibration.png")

In [ ]:
# ── Plot 4: ROC-AUC by Lifetime Stage ───────────────────────────────────────────
print("Generating Plot 4: ROC-AUC by Lifetime Stage...")
pct_labels = [f'{int(p*100)}%' for p in percentiles]
aucs_nt, aucs_wt, aucs_bl = [], [], []

for pct in percentiles:
    sub_s = test_shared[test_shared['target_percentile'] == pct]
    sub_t = test_trends[test_trends['target_percentile'] == pct]
    aucs_nt.append(roc_auc_score(sub_s['outcome'], sub_s['pred_proba_nt']))
    aucs_wt.append(roc_auc_score(sub_t['outcome'], sub_t['pred_proba_wt']))
    aucs_bl.append(roc_auc_score(sub_s['outcome'], sub_s['baseline']))

x     = np.arange(len(percentiles))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, aucs_nt, width, label='SVM no trends', color=COLORS['no_trends'],   alpha=0.85)
ax.bar(x,         aucs_wt, width, label='SVM+Trends',    color=COLORS['with_trends'], alpha=0.85)
ax.bar(x + width, aucs_bl, width, label='Baseline',       color=COLORS['baseline'],    alpha=0.85)
ax.set_xlabel('Lifetime Percentile')
ax.set_ylabel('ROC-AUC')
ax.set_title('ROC-AUC by Lifetime Stage — Shared Test Set')
ax.set_xticks(x)
ax.set_xticklabels(pct_labels)
ax.set_ylim(0.88, 0.98)
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'auc_by_lifetime_stage.png', dpi=150)
plt.show()
print("  Saved: auc_by_lifetime_stage.png")

In [ ]:
# ── Plot 5: Brier Score by Lifetime Stage ────────────────────────────────────────
print("Generating Plot 5: Brier Score by Lifetime Stage...")
bri_nt_list, bri_wt_list, bri_bl_list = [], [], []

for pct in percentiles:
    sub_s = test_shared[test_shared['target_percentile'] == pct]
    sub_t = test_trends[test_trends['target_percentile'] == pct]
    bri_nt_list.append(brier_score_loss(sub_s['outcome'], sub_s['pred_proba_nt']))
    bri_wt_list.append(brier_score_loss(sub_t['outcome'], sub_t['pred_proba_wt']))
    bri_bl_list.append(brier_score_loss(sub_s['outcome'], sub_s['baseline']))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(pct_labels, bri_nt_list, label='SVM no trends', color=COLORS['no_trends'],   lw=2, marker='o')
ax.plot(pct_labels, bri_wt_list, label='SVM+Trends',    color=COLORS['with_trends'], lw=2, marker='s')
ax.plot(pct_labels, bri_bl_list, label='Baseline',       color=COLORS['baseline'],    lw=2, marker='^', linestyle='--')
ax.set_xlabel('Lifetime Percentile')
ax.set_ylabel('Brier Score (lower is better)')
ax.set_title('Brier Score by Lifetime Stage — Shared Test Set')
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'brier_by_lifetime_stage.png', dpi=150)
plt.show()
print("  Saved: brier_by_lifetime_stage.png")

In [ ]:
# ── Plot 6: ROC-AUC by Category ─────────────────────────────────────────────────
print("Generating Plot 6: ROC-AUC by Category...")
x     = np.arange(len(categories))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - width, cat_aucs_nt, width, label='SVM no trends', color=COLORS['no_trends'],   alpha=0.85)
ax.bar(x,         cat_aucs_wt, width, label='SVM+Trends',    color=COLORS['with_trends'], alpha=0.85)
ax.bar(x + width, cat_aucs_bl, width, label='Baseline',       color=COLORS['baseline'],    alpha=0.85)
ax.set_xlabel('Category')
ax.set_ylabel('ROC-AUC')
ax.set_title('ROC-AUC by Category — Shared Test Set')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=30, ha='right')
ax.set_ylim(0.78, 0.98)
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'auc_by_category.png', dpi=150)
plt.show()
print("  Saved: auc_by_category.png")

In [ ]:
# ── Plot 7: Predicted Probability Distributions ──────────────────────────────────
print("Generating Plot 7: Predicted Probability Distributions...")
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (preds, y, title) in zip(axes, [
    (y_pred_proba_nt_shared, y_shared.values, 'SVM (no trends)'),
    (y_pred_proba_wt,        y_trends.values, 'SVM+Trends'),
    (baseline_shared.values, y_shared.values, 'Baseline'),
]):
    ax.hist(preds[y == 0], bins=50, alpha=0.6, label='outcome=0', color=COLORS['baseline'],  density=True)
    ax.hist(preds[y == 1], bins=50, alpha=0.6, label='outcome=1', color=COLORS['no_trends'], density=True)
    ax.set_xlabel('Predicted P(outcome=1)')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend()

plt.suptitle('Predicted Probability Distributions by Outcome', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'probability_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: probability_distributions.png")

In [ ]:
# ── Plot 8: Trends Feature Distributions by Outcome ─────────────────────────────
print("Generating Plot 8: Trends Feature Distributions by Outcome...")
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flatten(), ['trend_value', 'trend_ma4', 'trend_change_4w', 'trend_spike']):
    for outcome, color, label in [(0, COLORS['baseline'], 'outcome=0'),
                                   (1, COLORS['no_trends'], 'outcome=1')]:
        subset = test_trends[test_trends['outcome'] == outcome][col]
        ax.hist(subset, bins=40, alpha=0.6, label=label, color=color, density=True)
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.set_title(f'{col} by outcome')
    ax.legend()

plt.suptitle('Google Trends Feature Distributions by Outcome', fontsize=13)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'trends_feature_distributions.png', dpi=150)
plt.show()
print("  Saved: trends_feature_distributions.png")

In [ ]:
# ── Plot 9: Trends Improvement by Category ───────────────────────────────────────
print("Generating Plot 9: Trends Improvement by Category...")
improvements = [wt - nt for wt, nt in zip(cat_aucs_wt, cat_aucs_nt)]
colors_imp   = [COLORS['with_trends'] if v >= 0 else COLORS['baseline'] for v in improvements]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(categories, improvements, color=colors_imp, alpha=0.85)
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('Category')
ax.set_ylabel('ROC-AUC improvement (SVM+Trends minus SVM no trends)')
ax.set_title('Google Trends Improvement over Base SVM by Category')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'trends_improvement_by_category.png', dpi=150)
plt.show()
print("  Saved: trends_improvement_by_category.png")

print(f"\nAll plots saved to: {PLOT_DIR}")